# Homework: Grover's Algorithm
**Lecture 4.6**

---

**Overview**

In this assignment you will build, analyze, and extend Grover's algorithm. Part A walks you through constructing the oracle and diffusion operator from scratch. Part B asks you to trace the state evolution for a small system and verify the geometric picture. Part C scales the algorithm up and explores the overshoot problem. Part D is a partner activity inspired by the IBM Qiskit tutorial: you'll encode a secret target state, exchange circuits, and try to find each other's secrets.

**Total: 100 points**

**Due: Wednesday at midnight.**

---

**Implementation notes:**

- Use `Statevector` for exact verification and `AerSimulator` for shot-based experiments.
- When building oracles, remember **Qiskit qubit ordering**: qubit 0 is the rightmost bit in the ket. So $|1010\rangle$ means qubit 3 = 1, qubit 2 = 0, qubit 1 = 1, qubit 0 = 0.
- The helper functions `MCMTGate` and `GroverOperator` from `qiskit.circuit.library` are available, but in Parts A and B you should build the operators from gates so you understand how they work. You may use the library helpers in Parts C and D.
- Numerical results from the simulator may vary slightly due to shot noise. Use `np.isclose()` when comparing amplitudes.

In [ ]:
# Run this cell first — standard imports for the entire assignment
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit.circuit.library import MCMTGate, ZGate, GroverOperator
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import numpy as np
import matplotlib.pyplot as plt
import math

simulator = AerSimulator()

---
## Part A — Building the Grover Operator (25 pts)

### Background

Grover's algorithm uses two ingredients:

1. **The oracle** $Z_f$: flips the phase of marked states, $Z_f|x\rangle = (-1)^{f(x)}|x\rangle$.
2. **The diffusion operator** $D = H^{\otimes n}(2|0^n\rangle\langle 0^n| - I)H^{\otimes n}$: reflects all amplitudes about the mean.

In this part, you'll build both from basic gates (no `GroverOperator` yet).

### A1 (10 pts) — Build a phase oracle from gates

Write a function `build_oracle(target, n)` that takes a target bitstring (e.g., `"101"`) and the number of qubits $n$, and returns a `QuantumCircuit` implementing the phase oracle $Z_f$ that puts a $-1$ phase on $|\text{target}\rangle$ only.

**Recipe:**
1. Apply X gates to each qubit whose corresponding bit in the target is `'0'`.
2. Apply a multi-controlled Z gate using `MCMTGate(ZGate(), n-1, 1)`.
3. Apply the same X gates again to undo step 1.

**Test** your oracle on $n = 3$ with target `"101"`. Print the diagonal of the oracle's `Operator` matrix to verify that only $|101\rangle$ gets a phase of $-1$.

In [ ]:
def build_oracle(target, n):
    """
    Build a phase oracle that flips the phase of |target⟩.
    
    Parameters:
        target (str): bitstring of length n (e.g., "101")
        n (int): number of qubits
    Returns:
        QuantumCircuit: the phase oracle circuit
    """
    ### YOUR CODE HERE ###
    pass


# Test: oracle for |101⟩ on 3 qubits
oracle_test = build_oracle("101", 3)
oracle_test.draw(output="mpl", style="iqp")

In [ ]:
# Verify: print diagonal elements of the oracle matrix
op = Operator(oracle_test)
print("Oracle phases on each basis state:")
for i in range(2**3):
    bits = format(i, '03b')
    phase = np.real(op.data[i, i])
    marker = " ← MARKED" if phase < 0 else ""
    print(f"  |{bits}⟩: phase = {phase:+.0f}{marker}")

### A2 (10 pts) — Build the diffusion operator from gates

Write a function `build_diffusion(n)` that returns the $n$-qubit diffusion operator as a `QuantumCircuit`. Recall the circuit form:

$$D = H^{\otimes n} \cdot Z_{\text{OR}} \cdot H^{\otimes n}$$

where $Z_{\text{OR}} = 2|0^n\rangle\langle 0^n| - I$ flips the phase of every state *except* $|0\cdots 0\rangle$.

**Recipe for $Z_{\text{OR}}$:** Apply X to all qubits, then multi-controlled Z, then X to all qubits again.

**Verify** for $n = 3$: compute the `Operator` matrix of your diffusion operator and confirm that each entry of the matrix equals:

$$D_{ij} = \frac{2}{N} - \delta_{ij} \quad \text{where } N = 2^n$$

(i.e., $2/N$ on the off-diagonal, $2/N - 1$ on the diagonal).

In [ ]:
def build_diffusion(n):
    """
    Build the n-qubit diffusion operator D = H⊗n · Z_OR · H⊗n.
    
    Parameters:
        n (int): number of qubits
    Returns:
        QuantumCircuit: the diffusion operator circuit
    """
    ### YOUR CODE HERE ###
    pass


# Test: 3-qubit diffusion operator
diff_test = build_diffusion(3)
diff_test.draw(output="mpl", style="iqp")

In [ ]:
# Verify: check that D_ij = 2/N - delta_ij
n = 3
N = 2**n
D_matrix = Operator(diff_test).data

# Build expected matrix
expected = (2/N) * np.ones((N, N)) - np.eye(N)

print("Matrix matches expected form:", np.allclose(D_matrix, expected))

# Print a few entries for inspection
print(f"\nDiagonal entry D[0,0] = {np.real(D_matrix[0,0]):.4f}  (expected {2/N - 1:.4f})")
print(f"Off-diagonal D[0,1] = {np.real(D_matrix[0,1]):.4f}  (expected {2/N:.4f})")

### A3 (5 pts) — Conceptual: why two reflections make a rotation

The oracle $Z_f$ is a reflection about $|\omega^\perp\rangle$ and the diffusion operator $D$ is a reflection about $|s\rangle$. 

In 3–4 sentences, explain:
1. Why composing two reflections in a 2D plane produces a rotation.
2. What the rotation angle is in terms of $\theta = \arcsin(1/\sqrt{N})$.

*Hint:* Think about what happens when you reflect a vector across two different axes that make an angle $\theta$ with each other.

**Your answer (A3):**

*(double-click to edit)*

---
## Part B — State Evolution and the Geometric Picture (25 pts)

### B1 (15 pts) — Trace the amplitude evolution for $N = 8$

Use your `build_oracle` and `build_diffusion` functions from Part A to implement Grover's algorithm on $n = 3$ qubits ($N = 8$) with marked state $|101\rangle$.

**Do the following:**

1. Compute the optimal number of iterations: $t = \lfloor \frac{\pi}{4}\sqrt{N} \rfloor$.
2. Build the full Grover circuit (from gates, using your Part A functions). After each iteration of (oracle + diffusion), use `Statevector` to compute and print:
   - The amplitude of the marked state $|101\rangle$
   - The amplitude of one unmarked state (e.g., $|000\rangle$)
   - The probability of measuring the marked state
3. Print a summary table showing iteration number, marked amplitude, unmarked amplitude, and $P(\text{marked})$.

You should see the marked state's amplitude grow while the unmarked amplitudes shrink toward zero.

In [ ]:
### YOUR CODE HERE ###

n = 3
N = 2**n
target = "101"

# Compute optimal iterations
optimal_iter = math.floor(math.pi / 4 * math.sqrt(N))
print(f"n = {n}, N = {N}, optimal iterations = {optimal_iter}")
print()

# Build oracle and diffusion

# Evolve the state step by step
# Print amplitude table after each iteration

print(f"{'Iter':>4s}  {'a(marked)':>10s}  {'a(unmarked)':>12s}  {'P(marked)':>10s}")
print("-" * 42)

### B2 (10 pts) — Plot the geometric picture

Create a plot that shows the state vector's trajectory in the 2D subspace spanned by $|\omega\rangle$ and $|\omega^\perp\rangle$.

For each iteration $k = 0, 1, 2, \ldots, t$:
1. Compute the angle from the $|\omega^\perp\rangle$ axis: $\phi_k = (2k+1)\theta$ where $\theta = \arcsin(1/\sqrt{N})$.
2. Plot the state as a unit vector at angle $\phi_k$ from the horizontal.

Use $n = 4$ qubits ($N = 16$) so the rotation is more visible. Draw arrows from the origin for each iteration (use different colors), label the $|\omega\rangle$ and $|\omega^\perp\rangle$ axes, and include a legend.

**Also:** on the same plot or a second subplot, show the success probability $P = \sin^2((2k+1)\theta)$ vs. iteration number as a bar chart. Mark the optimal iteration in red.

In [ ]:
### YOUR CODE HERE ###

# Create a figure with 2 subplots: geometric picture + probability bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: geometric picture (arrows in the |ω⊥⟩, |ω⟩ plane)

# Right panel: success probability vs iteration number

plt.tight_layout()
plt.show()

---
## Part C — Scaling and the Overshoot Problem (25 pts)

### C1 (10 pts) — Grover's at scale with `GroverOperator`

Now you may use Qiskit's `GroverOperator` helper. Implement Grover's algorithm for $n = 5$ qubits ($N = 32$) with marked state `"10110"`.

1. Build the oracle using your `build_oracle` function (or the one from the lecture).
2. Use `GroverOperator(oracle)` to get the full Grover operator.
3. Compute the optimal iterations, build the circuit, and run with 2048 shots.
4. Plot the histogram. The marked state should clearly dominate.

**Print** the measured probability of the marked state (counts / total shots).

In [ ]:
### YOUR CODE HERE ###

n = 5
target = "10110"

# Build oracle, GroverOperator, compute optimal iterations

# Build full circuit, run 2048 shots, plot histogram

# Print measured probability of target state

### C2 (10 pts) — The overshoot problem

Using the same setup from C1 ($n = 5$, single marked state):

1. Run Grover's algorithm for $t = 0, 1, 2, \ldots, 15$ iterations. For each value of $t$, run 1024 shots and record the measured probability of the marked state.
2. Plot $P(\text{marked state})$ vs. $t$.
3. On the same plot, overlay the theoretical prediction: $P(t) = \sin^2((2t+1)\theta)$ where $\theta = \arcsin(1/\sqrt{N})$.

You should see the probability oscillate, confirming that more iterations can make the result *worse*.

**Label your axes.** Mark the optimal $t$ with a vertical dashed line.

In [ ]:
### YOUR CODE HERE ###

# Sweep over t = 0 to 15
# For each t: build circuit, run 1024 shots, record P(target)
# Also compute theoretical P(t) = sin^2((2t+1)*theta)

# Plot both on the same axes

### C3 (5 pts) — Conceptual: classical vs. quantum scaling

Answer the following in 3–5 sentences:

1. In classical computing, checking more entries in an unsorted database can never decrease your chance of finding the target. Why is this fundamentally different in Grover's algorithm?
2. Grover's gives a *quadratic* speedup ($O(\sqrt{N})$ vs. $O(N)$), not exponential. Explain why this is still significant — give a concrete example with numbers for a specific $N$.

**Your answer (C3):**

*(double-click to edit)*

---
## Part D — Partner Oracle Exchange (25 pts)

This activity is adapted from the [IBM Qiskit Grover's tutorial](../lecture_resources/grovers.ipynb). You and a partner will each secretly encode a target state into an oracle circuit, exchange the circuits as QPY files, and use Grover's algorithm to discover each other's secret.

### D1 (10 pts) — Encode your secret

1. Choose a number of qubits: $n = 4$ (so $N = 16$).
2. Choose a secret target bitstring (e.g., `"1101"`).
3. Build the oracle using your `build_oracle` function.
4. **Save** the oracle circuit to a QPY file.
5. Send the QPY file to your partner (email, messaging, shared folder, etc.).

**Important:** Tell your partner how many target states are encoded (just 1 in this case), but do NOT tell them which state it is.

In [ ]:
from qiskit import qpy

### YOUR CODE HERE ###

# Step 1: Choose your secret target
n = 4
my_secret = "????"  # Replace with your secret bitstring

# Step 2: Build the oracle

# Step 3: Save to QPY file
# with open("my_oracle.qpy", "wb") as f:
#     qpy.dump(oracle, f)

print(f"Oracle saved! Tell your partner: n = {n}, number of marked states = 1")
print("DO NOT tell them your target state!")

### D2 (10 pts) — Crack your partner's secret

1. Load your partner's oracle from their QPY file.
2. Build a Grover circuit using `GroverOperator` with the optimal number of iterations.
3. Run on the simulator with 2048 shots.
4. Plot the histogram and identify the marked state.

**Do NOT visualize or inspect your partner's oracle circuit** — that defeats the purpose of the query model!

In [ ]:
from qiskit import qpy

### YOUR CODE HERE ###

# Step 1: Load partner's oracle
# with open("partner_oracle.qpy", "rb") as f:
#     circuits = qpy.load(f)
# partner_oracle = circuits[0]

# Step 2: Build Grover circuit
num_marked = 1  # As told by your partner

# Step 3: Run and plot histogram

# Step 4: What is your partner's secret?

### D3 (5 pts) — Hamming weight oracle

Instead of marking a specific bitstring, create an oracle that marks *all* states with a given Hamming weight (number of 1s in the bitstring).

1. Write a function `hamming_oracle(n, weight)` that builds a phase oracle marking all $n$-qubit states with Hamming weight equal to `weight`. (*Hint:* enumerate all bitstrings with the correct weight and flip the phase on each one, using the same technique as `build_oracle`.)
2. Test with $n = 4$, weight = 2 (there should be $\binom{4}{2} = 6$ marked states).
3. Run Grover's algorithm with the correct number of marked states ($M = 6$) and 2048 shots. Plot the histogram.

You should see all 6 states with Hamming weight 2 appear with roughly equal probability.

In [ ]:
from itertools import combinations

def hamming_oracle(n, weight):
    """
    Build a phase oracle that marks all n-qubit states with the given Hamming weight.
    
    Parameters:
        n (int): number of qubits
        weight (int): target Hamming weight
    Returns:
        oracle (QuantumCircuit): the phase oracle
        marked_states (list): list of marked bitstrings
    """
    ### YOUR CODE HERE ###
    pass


# Test: n = 4, weight = 2
oracle_hw, marked_hw = hamming_oracle(4, 2)
print(f"Marked states (Hamming weight 2): {marked_hw}")
print(f"Number of marked states: {len(marked_hw)}")

In [ ]:
### YOUR CODE HERE ###

# Run Grover's with M = 6 marked states, n = 4
# Compute optimal iterations with M = len(marked_hw)
# Build circuit, run 2048 shots, plot histogram

---
**End of Homework**